# Distributed Systems Reality — Hands-On

**Software Engineering · Week 02+**

Offline simulations of idempotency keys, outbox relay, saga compensation, circuit breakers, backpressure, cache-stampede defense, consistent hashing, and capacity math.

## 0. Setup imports

In [ ]:
from dataclasses import dataclass
from collections import defaultdict, deque
import hashlib, json, bisect

## 1. Idempotency-key response cache

In [ ]:
@dataclass
class Record:
    params_hash: str
    status: int
    body: dict
    expires_at: float

class IdempotencyStore:
    def __init__(self, ttl=10):
        self.ttl = ttl
        self.records = {}
    def digest(self, payload):
        return hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()
    def handle(self, key, payload, now, handler):
        h = self.digest(payload)
        rec = self.records.get(key)
        if rec and rec.expires_at >= now:
            if rec.params_hash != h:
                return 409, {"error": "parameter mismatch"}
            return rec.status, {**rec.body, "cached": True}
        status, body = handler(payload)
        self.records[key] = Record(h, status, body, now + self.ttl)
        return status, body

store = IdempotencyStore()
handler = lambda p: (201, {"ticket_id": "T" + p["id"]})
print(store.handle("k", {"id": "1"}, 0, handler))
print(store.handle("k", {"id": "1"}, 1, handler))
print(store.handle("k", {"id": "2"}, 2, handler))

## 2. Transactional outbox simulation

In [ ]:
orders = {}
outbox = []
published = []

def create_order(order_id, amount):
    orders[order_id] = {"amount": amount, "status": "created"}
    outbox.append({"id": len(outbox)+1, "type": "OrderCreated", "order_id": order_id, "sent": False})

def relay_once():
    for event in outbox:
        if not event["sent"]:
            published.append((event["type"], event["order_id"]))
            event["sent"] = True

create_order("o1", 500)
relay_once(); relay_once()
print(orders, published)

## 3. Saga orchestration with compensation

In [ ]:
actions = []

def reserve_credit(): actions.append("reserve_credit")
def release_credit(): actions.append("release_credit")
def reserve_inventory(): actions.append("reserve_inventory")
def release_inventory(): actions.append("release_inventory")
def capture_payment(): raise RuntimeError("card declined")

def run_saga():
    completed = []
    try:
        reserve_credit(); completed.append(release_credit)
        reserve_inventory(); completed.append(release_inventory)
        capture_payment()
        return "committed"
    except Exception:
        for compensate in reversed(completed):
            compensate()
        return "compensated"

print(run_saga(), actions)

## 4. Circuit breaker with half-open state

In [ ]:
class CircuitBreaker:
    def __init__(self, threshold=2, cooldown=5):
        self.threshold = threshold; self.cooldown = cooldown
        self.failures = 0; self.state = "closed"; self.opened_at = None
    def call(self, now, fn):
        if self.state == "open" and now - self.opened_at < self.cooldown:
            return "blocked"
        if self.state == "open":
            self.state = "half-open"
        try:
            value = fn(); self.failures = 0; self.state = "closed"; return value
        except Exception:
            self.failures += 1
            if self.state == "half-open" or self.failures >= self.threshold:
                self.state = "open"; self.opened_at = now
            return "failed"

cb = CircuitBreaker()
bad = lambda: (_ for _ in ()).throw(TimeoutError())
for t in [0, 1, 2, 7]:
    print(t, cb.call(t, bad), cb.state)

## 5. Token bucket backpressure

In [ ]:
class TokenBucket:
    def __init__(self, capacity, refill):
        self.capacity = capacity; self.refill = refill; self.tokens = capacity; self.updated = 0
    def allow(self, now):
        self.tokens = min(self.capacity, self.tokens + max(0, now - self.updated) * self.refill)
        self.updated = now
        if self.tokens >= 1:
            self.tokens -= 1; return True
        return False

bucket = TokenBucket(3, 1)
print([bucket.allow(t) for t in [0, 0, 0, 0, 1, 1.1, 2]])

## 6. Cache stampede request coalescing

In [ ]:
cache = {}
inflight = set()
loads = defaultdict(int)

def get_or_start_load(key):
    if key in cache:
        return "hit", cache[key]
    if key in inflight:
        return "coalesced", None
    inflight.add(key); loads[key] += 1
    return "load_started", None

print([get_or_start_load("tenant:acme") for _ in range(5)])
cache["tenant:acme"] = {"plan": "enterprise"}; inflight.remove("tenant:acme")
print(get_or_start_load("tenant:acme"), "loads", dict(loads))

## 7. Consistent hashing ring

In [ ]:
class HashRing:
    def __init__(self, nodes, replicas=3):
        pairs = []
        for node in nodes:
            for r in range(replicas):
                h = int(hashlib.md5(f"{node}:{r}".encode()).hexdigest(), 16)
                pairs.append((h, node))
        self.ring = sorted(pairs)
        self.points = [p[0] for p in self.ring]
    def node_for(self, key):
        h = int(hashlib.md5(key.encode()).hexdigest(), 16)
        i = bisect.bisect(self.points, h) % len(self.ring)
        return self.ring[i][1]

ring = HashRing(["cache-a", "cache-b", "cache-c"])
print({k: ring.node_for(k) for k in ["u1", "u2", "u3", "u4"]})

## 8. Capacity math

In [ ]:
qps = 2_000
payload_kb = 20
bandwidth_mb_s = qps * payload_kb / 1024
events_per_day = 50_000_000
event_kb = 2
storage_gb_day = events_per_day * event_kb / 1024 / 1024
provider_p99_ms = 180
budget_ms = 300
remaining = budget_ms - provider_p99_ms
print(f"bandwidth≈{bandwidth_mb_s:.1f} MB/s storage≈{storage_gb_day:.1f} GB/day remaining_budget={remaining}ms")

## Exercises
1. Add parameter-hash storage to an HTTP middleware shape.
2. Add a DLQ after three failed outbox publishes.
3. Change the consistent-hash ring from 3 to 100 replicas and compare movement.
4. Compute worker count for 100 QPS with 250 ms service time and 2x headroom.

## Links
- Literature note: `02 Literature Notes/Software Engineering/Distributed Systems Reality`
- Snippets: `04 Code Snippets/Software Engineering/SE Week 02+ Idempotency Key Middleware Simulation`, `.../SE Week 02+ Token Bucket Circuit Breaker`
- MOC: `06 Maps of Content/Software Engineering Concepts`